In [ ]:
import pandas as pd
import requests
import json
import time
import re
import unicodedata
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
df = pd.read_json("companies.jsonl", lines = True)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [ ]:
def norm_f(x):
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        return " ".join(str(i) for i in x)
    if isinstance(x, dict):
        return " ".join(str(v) for v in x.values())
    if x is None:
        return ""
    else:
        return str(x)

In [ ]:
def company_profile(line):
    info=[
        line["website"],
        line["operational_name"],
        line["year_founded"],
        line["address"],
        line["employee_count"],
        line["revenue"],
        line["primary_naics"],
        line["description"],
        line["business_model"],
        line["target_markets"],
        line["core_offerings"],
        line["is_public"],
        line["secondary_naics"]
         ]
    text = " ".join(norm_f(i) for i in info)
    return text 

In [ ]:
df["company_profiles"] = df.apply(company_profile, axis=1)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
company_vectors = model.encode(df["company_profiles"].tolist())

In [ ]:
with open("queries_examples.txt", "r", encoding="utf-8") as f:
    queries = [line.strip() for line in f if line.strip()]

In [ ]:
def call_groq_json(prompt):
    """For structured JSON object responses, not arrays."""
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [
            {"role": "system", "content": "You are a query parser. Return ONLY a valid JSON object. No text, no markdown, no explanation."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }
    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    if "choices" in data:
        return data["choices"][0]["message"]["content"]
    else:
        print("Error:", data)
        return "{}"

In [ ]:
def parse_query(query):
    prompt = f"""Extract search intent from this query: "{query}"
Return ONLY JSON with these fields:
{{
  "industry_keywords": ["expand with synonyms and related terms, e.g. for logistics include: logistic, logistics, freight, transport, warehouse, supply chain, shipping, courier"],
  "location": "country or city or null",
  "company_size": "small/medium/large or null",
  "other_filters": []
}}"""
    return call_groq_json(prompt)

In [ ]:
def get_candidates(query, parsed_query, df, company_vectors, threshold=0.3, top_n=20):
    query_vector = model.encode([query])
    scores = cosine_similarity(query_vector, company_vectors).flatten()
    df["embedding_scores"] = scores
    top_candidates = df.nlargest(top_n, "embedding_scores")
    above_threshold = top_candidates[top_candidates["embedding_scores"] >= threshold]
    return above_threshold

In [ ]:
def remove_diacritics(text):
    if not isinstance(text, str):
        return text
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

In [ ]:
def build_companies_text(top_candidates):
    companies_list = []
    for comp_id, line in top_candidates.iterrows():
        naics = line["primary_naics"]
        if isinstance(naics, dict):
            naics_label = naics.get("label", "")
        elif isinstance(naics, str):
            naics_label = naics
        else:
            naics_label = ""
        secondary = line["secondary_naics"]
        if isinstance(secondary, list):
            secondary_label = ", ".join(
                s.get("label", "") if isinstance(s, dict) else str(s) 
                for s in secondary
            )
        else:
            secondary_label = ""
        company = {
            "id": comp_id,
            "name": remove_diacritics(str(line["operational_name"])),
            "website": str(line["website"]),
            "year_founded": line["year_founded"],
            "address": remove_diacritics(str(line["address"])),
            "employees": line["employee_count"],
            "revenue": line["revenue"],
            "naics": naics_label,
            "secondary_naics": secondary_label,
            "description": remove_diacritics(str(line["description"]))[:300],
            "business_model": line["business_model"],
            "target_markets": line["target_markets"],
            "is_public": line["is_public"]
        }
        companies_list.append(company)
    return json.dumps(companies_list)

In [ ]:
def build_prompt(query, top_candidates, parsed_query):
    companies_text = build_companies_text(top_candidates)
    keywords = parsed_query.get("industry_keywords", [])
    return f"""
User query: {query}

The following terms represent valid matches for this query:
{json.dumps(keywords)}

Use a BROAD definition — any company whose primary or secondary business 
relates to these terms is a valid match.

You MUST evaluate and return EVERY company in the list.
Total companies to evaluate: {len(top_candidates)}

Score rules:
- 1.0 = primary business directly matches the query
- 0.5 = secondary or partial match
- 0.0 = no relation at all

Return ONLY a valid JSON array with ALL {len(top_candidates)} companies:
[{{"id": 0, "name": "Company Name", "score": 1.0}}]

Candidate companies:
{companies_text}
"""

In [ ]:
def call_groq(prompt):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [
            {"role": "system", "content": "You are a company qualification engine. Return ONLY valid JSON arrays. No text, no markdown, no explanation."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }
    for attempt in range(3):
        response = requests.post(url, headers=headers, json=payload)
        data = response.json()
        if "choices" in data:
            return data["choices"][0]["message"]["content"]
        elif response.status_code == 429:
            print(f"Rate limit hit, waiting 30s...")
            time.sleep(30)
        else:
            print("Error:", data)
            return "[]"
    return "[]"

In [ ]:
for query in queries:
    print(f"\nQuery: {query}")
    parsed_query = json.loads(parse_query(query))
    time.sleep(5)
    top_candidates = get_candidates(query, parsed_query, df, company_vectors)
    if len(top_candidates) == 0:
        print("No candidates found")
        continue
    prompt = build_prompt(query, top_candidates, parsed_query)
    response_text = call_groq(prompt).strip()
    try:
        results = json.loads(response_text)
        if isinstance(results, dict):
            unwrapped = next((v for v in results.values() if isinstance(v, list)), None)
            results = unwrapped if unwrapped else [results]
    except json.JSONDecodeError:
        match = re.search(r"\[.*?\]", response_text, re.DOTALL)
        results = json.loads(match.group()) if match else []
    final_results = [item for item in results if item["score"] > 0.0]
    print(f"\nResults:")
    print(f"{'ID':<6} {'Score':<10} {'Company':<40} {'Verdict'}")
    for item in final_results:
        verdict = "Full Match" if item["score"] == 1.0 else "Partial Match"
        print(f"{item['id']:<6} {item['score']:<10} {item['name']:<40} {verdict}")
    print(f"\nTotal: {len(final_results)} companies found")
    time.sleep(10)